# RAG (2020)
---
[[paper]](https://arxiv.org/abs/2005.11401)<br>
RAG = Retrieval-Augmented Generation

__RAG__ — это архитектура, объединяющая параметрическую память (предварительно обученную seq2seq модель) и непараметрическую память (плотный векторный индекс документов, например, Wikipedia), позволяющая генерировать ответы на основе актуальных внешних данных без необходимости переобучения всей модели.

__Постановка задачи__<br>
Решается задача генерации текста для Knowledge-Intensive Tasks (ответы на вопросы, фактологическая проверка). Модель должна принять запрос $x$ и сгенерировать текст $y$, используя при этом внешнее хранилище знаний.

__Мотивация__<br>
LLM (Large Language Models) "запоминают" факты в своих весах (параметрическая память). Однако у этого подхода есть три проблемы: 
1. Невозможно легко обновить или исправить знания.
2. Склонность к галлюцинациям при ответе на узкоспециализированные вопросы. 
3. Высокая стоимость дообучения для расширения базы знаний. 
Идея RAG состоит в том, чтобы дать модели "открытую книгу", в которую она может подглядывать во время генерации.

__Существующие подходы__<br>
На момент публикации (2020) основными альтернативами были:
- Closed-book QA (модели T5 (2019) или BART (2019)): полагаются только на веса. Точность на сложных фактах (например, Natural Questions) была ниже, чем у систем с поиском.
- Open-domain QA с экстрактивными моделями (DPR (2020)): находят документ и выделяют в нем конкретный фрагмент текста (span) как ответ. Они не умеют синтезировать ответ из нескольких источников или генерировать связный текст.
- REALM (2020): первая попытка совместного обучения поиска и генерации, но она была сфокусирована на Masked Language Model и менее эффективна для полноценной генерации текста.

__Идея__<br>
Авторы предложили рассматривать процесс генерации как работу с латентной переменной $z$ (релевантные документы). Вместо того чтобы сразу предсказывать $y$ по $x$, модель сначала ищет топ-K документов $z$, а затем использует их как дополнительный контекст. Ключевая новизна — возможность сквозного обучения (End-to-End) всей системы: и поискового энкодера (Retriever), и генератора (Generator).

__Архитектура__<br>
Система состоит из двух основных компонентов:
1. Retriever: использует Dense Passage Retrieval (DPR). Это Two-Tower архитектура на базе BERT (2018). Один энкодер кодирует запрос, другой — документы. Поиск происходит через Maximum Inner Product Search (MIPS) в векторном пространстве.
2. Generator: предобученная seq2seq модель BART. Она получает на вход конкатенацию запроса $x$ и найденного документа $z$.
3. Индекс: векторное хранилище (обычно на базе FAISS), содержащее эмбеддинги 21 млн документов из Wikipedia.

__Алгоритм обучения__<br>
Обучение минимизирует Negative Log Likelihood (NLL) для целевой последовательности $y$. 
1. Retriever и Generator инициализируются предобученными весами (DPR и BART).
2. Для каждого примера из обучающей выборки выполняется поиск документов.
3. Градиенты пробрасываются через Generator обратно в Query Encoder компонента Retriever. Это позволяет модели "научиться" лучше кодировать запросы так, чтобы MIPS выдавал более полезные документы для итоговой генерации. Document Encoder обычно остается замороженным для экономии ресурсов.

__Алгоритм инференса__<br>
Предложено два варианта реализации:
1. RAG-Sequence: Для всей генерации выбирается фиксированный набор из $K$ документов. Модель генерирует распределение для каждой последовательности отдельно (для каждого документа), а затем вероятности суммируются.
2. RAG-Token: Для каждого генерируемого токена модель может опираться на разные документы. На каждом шаге декодирования рассчитывается взвешенное распределение вероятностей по всем $K$ документам. Это позволяет комбинировать факты из разных источников в одном предложении.

__Результаты__<br>
- На датасете Natural Questions (Open-domain QA) RAG-Sequence набрал 44.5% Exact Match, что на 4.4пп выше, чем у модели T5 (11B параметров), при этом RAG использует значительно меньше параметров.
- В задачах генерации (Jeopardy question generation) RAG показал более высокую фактологическую точность по сравнению с BART (параметрическая память), так как BART чаще путал сущности и даты.
- Модель продемонстрировала способность к обновлению знаний: достаточно заменить индекс документов на новый, и RAG начнет отвечать на вопросы по актуальной базе без дообучения весов.

## 📝 Критический анализ

# RAG (2020)
---
[[paper]](https://arxiv.org/abs/2005.11401)<br>
RAG = Retrieval-Augmented Generation

**RAG** — это архитектура, объединяющая параметрическую память (предварительно обученную seq2seq модель) и непараметрическую память (векторный индекс документов, например, Wikipedia), для генерации ответов на основе актуальных данных без переобучения модели.

## Постановка задачи
Решается задача генерации текста для Knowledge-Intensive Tasks (ответы на вопросы, фактологическая проверка). Модель принимает запрос $x$ и генерирует текст $y$, используя внешнее хранилище знаний.

## Мотивация
LLM "запоминают" факты в весах, что приводит к проблемам: сложность обновления знаний, галлюцинации при узкоспециализированных вопросах и высокая стоимость дообучения. RAG предлагает "открытую книгу" для модели.

## Существующие подходы
На момент публикации (2020) альтернативами были:
- Closed-book QA (T5 (2019), BART (2019)): полагаются на веса, точность ниже на сложных фактах.
- Open-domain QA с экстрактивными моделями (DPR (2020)): выделяют фрагменты текста, но не синтезируют ответы.
- REALM (2020): совместное обучение поиска и генерации, но менее эффективно для генерации текста.

## Идея
Процесс генерации рассматривается как работа с латентной переменной $z$ (релевантные документы). Модель сначала ищет топ-K документов $z$, затем использует их как контекст. Новизна — сквозное обучение (End-to-End) системы: и Retriever, и Generator.

## Архитектура
Система состоит из:
1. **Retriever**: использует Dense Passage Retrieval (DPR) с Two-Tower архитектурой на базе BERT (2018). Поиск через Maximum Inner Product Search (MIPS).
2. **Generator**: предобученная seq2seq модель BART, получает конкатенацию запроса $x$ и документа $z$.
3. **Индекс**: векторное хранилище (обычно FAISS), содержащее эмбеддинги 21 млн документов из Wikipedia.

<img src="img/img.png" width=500>

## Алгоритм обучения
Минимизируется Negative Log Likelihood (NLL) для целевой последовательности $y$.
1. Инициализация Retriever и Generator предобученными весами (DPR и BART).
2. Поиск документов для каждого примера из обучающей выборки.
3. Градиенты пробрасываются через Generator в Query Encoder Retriever. Document Encoder остается замороженным.

## Алгоритм инференса
1. **RAG-Sequence**: фиксированный набор из $K$ документов для всей генерации. Вероятности суммируются.
2. **RAG-Token**: для каждого токена модель опирается на разные документы. Вероятности рассчитываются по всем $K$ документам.

## Результаты
- На Natural Questions (Open-domain QA) RAG-Sequence набрал 44.5% Exact Match, на 4.4пп выше T5 (11B параметров), с меньшим числом параметров.
- В генерации (Jeopardy question generation) RAG показал более высокую фактологическую точность по сравнению с BART.
- Модель обновляет знания заменой индекса документов, отвечая на актуальные вопросы без дообучения.

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример реализации Retrieval-Augmented Generation (RAG) с использованием библиотеки Hugging Face Transformers.
# Мы будем использовать предварительно обученные модели DPR и BART для иллюстрации основных концепций RAG.

from transformers import DPRQuestionEncoder, DPRContextEncoder, DPRQuestionEncoderTokenizer, DPRContextEncoderTokenizer
from transformers import BartForConditionalGeneration, BartTokenizer
import torch
import faiss
import numpy as np

# 1. Инициализация моделей и токенизаторов
question_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
context_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
context_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")

bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")

# 2. Создание векторного индекса для документов (например, Wikipedia)
# Для простоты, используем игрушечный набор документов
documents = [
    "The Eiffel Tower is located in Paris.",
    "The Great Wall of China is visible from space.",
    "Python is a programming language."
]

# Кодируем документы с помощью контекстного энкодера
document_embeddings = []
for doc in documents:
    inputs = context_tokenizer(doc, return_tensors="pt")
    embeddings = context_encoder(**inputs).pooler_output
    document_embeddings.append(embeddings.detach().numpy())

# Создаем FAISS индекс для быстрого поиска
dimension = document_embeddings[0].shape[1]
index = faiss.IndexFlatIP(dimension)  # Используем скалярное произведение для поиска
index.add(np.vstack(document_embeddings))

# 3. Функция для поиска релевантных документов
def retrieve_documents(query, top_k=2):
    query_inputs = question_tokenizer(query, return_tensors="pt")
    query_embedding = question_encoder(**query_inputs).pooler_output.detach().numpy()
    
    # Поиск топ-K релевантных документов
    distances, indices = index.search(query_embedding, top_k)
    return [documents[i] for i in indices[0]]

# 4. Генерация ответа с использованием BART
def generate_answer(query):
    # Находим релевантные документы
    relevant_docs = retrieve_documents(query)
    
    # Конкатенируем запрос и документы
    context = " ".join(relevant_docs)
    input_text = f"Query: {query} Context: {context}"
    
    # Генерация ответа
    inputs = bart_tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = bart_model.generate(inputs["input_ids"], num_beams=4, max_length=50, early_stopping=True)
    return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Пример использования
query = "Where is the Eiffel Tower located?"
answer = generate_answer(query)
print(f"Query: {query}\nAnswer: {answer}")

# В этом примере мы продемонстрировали основные этапы RAG:
# 1. Использование DPR для поиска релевантных документов.
# 2. Использование BART для генерации ответа на основе найденных документов.
# 3. Возможность обновления знаний путем изменения индекса документов.
```

Этот код иллюстрирует основные концепции RAG, такие как использование DPR для поиска релевантных документов и BART для генерации ответов. Мы также показали, как можно обновлять знания модели, изменяя индекс документов.